<a href="https://colab.research.google.com/github/Bonesha-5/Group4_Formative3_Assignment/blob/main/bayesian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries and Loading the Data

In [2]:
import csv

In [3]:
with open("IMDB Dataset.csv", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    reviews = list(reader)

print(f"Total reviews: {len(reviews)}")

Total reviews: 24515


## Selection Of Keywords

In [4]:
positive_keywords = [
    "excellent",
    "superb",
    "masterpiece",
    "outstanding"
]

negative_keywords = [
    "terrible",
    "awful",
    "horrible",
    "disappointing"
]

keywords = positive_keywords + negative_keywords

print(keywords)

['excellent', 'superb', 'masterpiece', 'outstanding', 'terrible', 'awful', 'horrible', 'disappointing']


**Calculating prior probabilities**

In [5]:
# PRIOR AND LIKELIHOOD CALCULATIONS

# Keep only reviews with valid sentiment labels
valid_reviews = []

for review in reviews:
    sentiment_val = review.get("sentiment")
    sentiment = (sentiment_val or "").strip().lower()

    if sentiment in ["positive", "negative"]:
        valid_reviews.append(review)

reviews = valid_reviews

# Count the total number of valid reviews
total_reviews = len(reviews)

# Count positive and negative reviews
positive_reviews = 0
negative_reviews = 0

for review in reviews:
    sentiment_val = review.get("sentiment")
    sentiment = (sentiment_val or "").strip().lower()

    if sentiment == "positive":
        positive_reviews += 1
    elif sentiment == "negative":
        negative_reviews += 1

# Calculate prior probabilities
prior_positive = positive_reviews / total_reviews if total_reviews > 0 else 0
prior_negative = negative_reviews / total_reviews if total_reviews > 0 else 0

# Display results
print("Total reviews:", total_reviews)
print("Positive reviews:", positive_reviews)
print("Negative reviews:", negative_reviews)

print("Prior P(Positive):", round(prior_positive, 4))
print("Prior P(Negative):", round(prior_negative, 4))

Total reviews: 24514
Positive reviews: 12223
Negative reviews: 12291
Prior P(Positive): 0.4986
Prior P(Negative): 0.5014


**Calculating keyword likelihood**

In [6]:
# Calculate likelihoods for each selected keyword

import re

# Store likelihood results
likelihood_results = []

for keyword in keywords:

    positive_count = 0
    negative_count = 0

    for review in reviews:
        sentiment = review["sentiment"].strip().lower()
        text = review["review"].lower()

        # Check whether the whole keyword appears in the review
        if re.search(r"\b" + re.escape(keyword) + r"\b", text):

            if sentiment == "positive":
                positive_count += 1

            elif sentiment == "negative":
                negative_count += 1

    # Calculate likelihoods for both classes
    likelihood_positive = positive_count / positive_reviews
    likelihood_negative = negative_count / negative_reviews

    # Store the calculated values
    likelihood_results.append([
        keyword,
        positive_count,
        negative_count,
        likelihood_positive,
        likelihood_negative
    ])

**Prior probabilities and likelihood results**

In [7]:
# Display the calculated prior probabilities

print("PRIOR PROBABILITIES")
print("-" * 40)

print(f"P(Positive): {prior_positive:.4f}")
print(f"P(Negative): {prior_negative:.4f}")

print("\nKEYWORD LIKELIHOODS")
print("-" * 85)

# Display likelihoods in a table
print(
    f"{'Keyword':<18}"
    f"{'Positive Count':<18}"
    f"{'Negative Count':<18}"
    f"{'P(W|Positive)':<16}"
    f"{'P(W|Negative)':<16}"
)

print("-" * 85)

for row in likelihood_results:
    keyword, pos_count, neg_count, likelihood_pos, likelihood_neg = row

    print(
        f"{keyword:<18}"
        f"{pos_count:<18}"
        f"{neg_count:<18}"
        f"{likelihood_pos:<16.4f}"
        f"{likelihood_neg:<16.4f}"
    )

PRIOR PROBABILITIES
----------------------------------------
P(Positive): 0.4986
P(Negative): 0.5014

KEYWORD LIKELIHOODS
-------------------------------------------------------------------------------------
Keyword           Positive Count    Negative Count    P(W|Positive)   P(W|Negative)   
-------------------------------------------------------------------------------------
excellent         1428              332               0.1168          0.0270          
superb            480               88                0.0393          0.0072          
masterpiece       413               157               0.0338          0.0128          
outstanding       311               75                0.0254          0.0061          
terrible          197               1151              0.0161          0.0936          
awful             124               1280              0.0101          0.1041          
horrible          144               844               0.0118          0.0687          
disappoint

Marginal probability (how often each keyword appears in all reviews)

In [19]:
# Calculate the marginal probability P(word) for each keyword
marginal_results = []

print(f"{'Keyword':<16}{'Reviews':>10}{'P(word)':>10}")
print("-" * 36)

for row in likelihood_results:
    keyword, pos_count, neg_count, likelihood_pos, likelihood_neg = row

    # Reviews that contain the keyword (positive + negative)
    word_count = pos_count + neg_count
    marginal = word_count / total_reviews

    marginal_results.append([keyword, word_count, marginal])

    print(f"{keyword:<16}{word_count:>10}{marginal:>10.4f}")

Keyword            Reviews   P(word)
------------------------------------
excellent             1760    0.0718
superb                 568    0.0232
masterpiece            570    0.0233
outstanding            386    0.0157
terrible              1348    0.0550
awful                 1404    0.0573
horrible               988    0.0403
disappointing          381    0.0155


Posterior probability (how likely a review is positive when it contains the keyword.)

In [18]:
# Bayes' theorem: P(Positive | word) = P(word | Positive) * P(Positive) / P(word)
posterior_results = []

for i in range(len(likelihood_results)):
    keyword = likelihood_results[i][0]
    likelihood_pos = likelihood_results[i][3]
    marginal = marginal_results[i][2]

    posterior = (likelihood_pos * prior_positive) / marginal

    posterior_results.append([keyword, posterior])

    print(f"P(Positive | {keyword}) = {posterior:.4f}")

P(Positive | excellent) = 0.8114
P(Positive | superb) = 0.8451
P(Positive | masterpiece) = 0.7246
P(Positive | outstanding) = 0.8057
P(Positive | terrible) = 0.1461
P(Positive | awful) = 0.0883
P(Positive | horrible) = 0.1457
P(Positive | disappointing) = 0.2021


Check (compare the Bayes result with a direct count.)

In [20]:
# Check: Bayes result should equal positive reviews with the word / all reviews with the word
print(f"{'Keyword':<16}{'Bayes':>8}{'Direct':>8}{'Match':>8}")
print("-" * 40)

for i in range(len(likelihood_results)):
    keyword, pos_count, neg_count = likelihood_results[i][0:3]

    direct = pos_count / (pos_count + neg_count)
    bayes = posterior_results[i][1]
    match = "Yes" if abs(bayes - direct) < 1e-9 else "No"

    print(f"{keyword:<16}{bayes:>8.4f}{direct:>8.4f}{match:>8}")

Keyword            Bayes  Direct   Match
----------------------------------------
excellent         0.8114  0.8114     Yes
superb            0.8451  0.8451     Yes
masterpiece       0.7246  0.7246     Yes
outstanding       0.8057  0.8057     Yes
terrible          0.1461  0.1461     Yes
awful             0.0883  0.0883     Yes
horrible          0.1457  0.1457     Yes
disappointing     0.2021  0.2021     Yes


In [16]:
print("BAYES' THEOREM RESULTS")
print("-" * 61)
print(
    f"{'Keyword':<16}"
    f"{'Prior':<10}"
    f"{'Likelihood':<14}"
    f"{'Marginal':<12}"
    f"{'Posterior':<12}"
)
print("-" * 61)

for i in range(len(likelihood_results)):
    keyword = likelihood_results[i][0]
    likelihood_pos = likelihood_results[i][3]
    marginal = marginal_results[i][2]
    posterior = posterior_results[i][1]

    print(
        f"{keyword:<16}"
        f"{prior_positive:<10.4f}"
        f"{likelihood_pos:<14.4f}"
        f"{marginal:<12.4f}"
        f"{posterior:<12.4f}"
    )

BAYES' THEOREM RESULTS
-------------------------------------------------------------
Keyword         Prior     Likelihood    Marginal    Posterior   
-------------------------------------------------------------
excellent       0.4986    0.1168        0.0718      0.8114      
superb          0.4986    0.0393        0.0232      0.8451      
masterpiece     0.4986    0.0338        0.0233      0.7246      
outstanding     0.4986    0.0254        0.0157      0.8057      
terrible        0.4986    0.0161        0.0550      0.1461      
awful           0.4986    0.0101        0.0573      0.0883      
horrible        0.4986    0.0118        0.0403      0.1457      
disappointing   0.4986    0.0063        0.0155      0.2021      
